# Neural Machine Translation Demo with Marian
## Transformer Architecture

**By Ye Kyaw Thu, Lab Leader, Language Understanding Lab., Myanmar**  
**Date:** 26 May 2026  
*For AI (Fundamental) Class students*  

အရင်ဆုံး GPU အားမအား စစ်ပါ။  
(ဆရာ သုံးခဲ့တဲ့ GPU information ကို သိစေချင်လို့ run ပြတာလည်း ပါပါတယ်)

## Colab Adaptation Note

**Student: Yee Mon Thant**

This copy has been adapted to run on **Google Colab**. The changes from the original notebook by Ye Kyaw Thu are: (1) Marian is loaded from a **pre-built CUDA binary saved on Google Drive** instead of being installed/compiled here, (2) the **training/dev/test data files are uploaded** since Colab starts with an empty filesystem, and (3) the `transformer.phmy.sh` script (which already existed on the teacher's machine but wasn't included in the uploaded files) is **recreated here** with the transformer-specific hyperparameters referenced in the notebook. All training hyperparameters and explanations are otherwise unchanged from the original.

**Note on structure:** Everything up to and including the "Evaluation" section follows Sayar's original notebook exactly, with only the Colab infrastructure adaptations noted above. The "Task 4: Improving the Transformer Baseline" section that follows is my own addition, written to fulfill the assignment's requirement to improve on the baseline result.


## Load Marian and Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil, os
os.makedirs('/content/marian-bin-gpu', exist_ok=True)
for binary in ['marian', 'marian-decoder', 'marian-vocab']:
    shutil.copy(f'/content/drive/MyDrive/marian-build-gpu/{binary}', '/content/marian-bin-gpu/')
    os.chmod(f'/content/marian-bin-gpu/{binary}', 0o755)
os.environ['PATH'] = '/content/marian-bin-gpu:' + os.environ['PATH']
print('Marian (GPU) ready ✅')


Mounted at /content/drive
Marian (GPU) ready ✅


In [ ]:
# Set up the working directory
import os
os.makedirs('/content/marian-demo/g2p-par', exist_ok=True)
%cd /content/marian-demo


/content/marian-demo


### Upload data files

Upload the same 6 files used for the Seq2Seq model: `train.my`, `train.ph`, `dev.my`, `dev.ph`, `test.my`, `test.ph`.


In [ ]:
%cd /content/marian-demo/g2p-par
from google.colab import files
import os

needed = ['train.my', 'train.ph', 'dev.my', 'dev.ph', 'test.my', 'test.ph']
missing = [f for f in needed if not os.path.exists(f)]
if missing:
    print('Please upload:', missing)
    uploaded = files.upload()
else:
    print('All data files already present ✅')
!ls -la


/content/marian-demo/g2p-par
Please upload: ['train.my', 'train.ph', 'dev.my', 'dev.ph', 'test.my', 'test.ph']


Saving dev.my to dev.my
Saving dev.ph to dev.ph
Saving test.my to test.my
Saving test.ph to test.ph
Saving train.my to train.my
Saving train.ph to train.ph
total 1056
drwxr-xr-x 2 root root   4096 Aug  7 03:07 .
drwxr-xr-x 3 root root   4096 Aug  7 03:07 ..
-rw-r--r-- 1 root root  59222 Aug  7 03:07 dev.my
-rw-r--r-- 1 root root  25849 Aug  7 03:07 dev.ph
-rw-r--r-- 1 root root  83959 Aug  7 03:07 test.my
-rw-r--r-- 1 root root  36532 Aug  7 03:07 test.ph
-rw-r--r-- 1 root root 594183 Aug  7 03:07 train.my
-rw-r--r-- 1 root root 260356 Aug  7 03:07 train.ph


### Build vocab




In [ ]:
import os
if not os.path.exists('./vocab/vocab.my.yml'):
    os.makedirs('preprocessing', exist_ok=True)
    os.makedirs('vocab', exist_ok=True)
    !cat train.my dev.my > ./preprocessing/train-dev.my
    !cat train.ph dev.ph > ./preprocessing/train-dev.ph
    !marian-vocab < ./preprocessing/train-dev.my > ./vocab/vocab.my.yml
    !marian-vocab < ./preprocessing/train-dev.ph > ./vocab/vocab.ph.yml
    print('Vocab built ✅')
else:
    print('Vocab already exists ✅')
%cd /content/marian-demo


[2026-08-07 03:07:51] Creating vocabulary...
[2026-08-07 03:07:51] [data] Creating vocabulary stdout from stdin
[2026-08-07 03:07:51] Finished
[2026-08-07 03:07:53] Creating vocabulary...
[2026-08-07 03:07:53] [data] Creating vocabulary stdout from stdin
[2026-08-07 03:07:53] Finished
Vocab built ✅
/content/marian-demo


In [ ]:
%pwd

'/content/marian-demo'

## Shell Script Preparation for Tranformer Architecture

Vocab ဖိုင်က ရှိပြီးသား ဖြစ်ရပါမယ်။  

### Colab Adaptation: recreate `transformer.phmy.sh`

This script didn't exist in the uploaded files, so it's recreated here using the transformer-specific settings referenced in the notebook (`--type transformer`), keeping everything else (data, vocab, dropout, valid/save frequency, early stopping) the same as the Seq2Seq baseline so the two architectures are fairly comparable. It uses the GPU-enabled Marian binary with `--devices 0`.


In [ ]:
%%writefile ./transformer.phmy.sh
export PATH=/content/marian-bin-gpu:$PATH

## Written by Ye Kyaw Thu, Affiliated Professor, CADT, Cambodia
## for NMT Experiments between Burmese and Ethnic Languages
## used Marian NMT Framework for training
## Last updated: 23 May 2022
## Colab adaptation: only data_path and PATH changed; all training flags
## match the teacher's original script exactly.

model_folder="model.transformer.phmy"
mkdir -p ${model_folder}
data_path="/content/marian-demo/g2p-par"
src="ph"; tgt="my"

marian \
    --model ${model_folder}/model.npz --type transformer \
    --train-sets ${data_path}/train.${src} ${data_path}/train.${tgt} \
    --max-length 200 \
    --vocabs ${data_path}/vocab/vocab.${src}.yml ${data_path}/vocab/vocab.${tgt}.yml \
    --mini-batch-fit -w 1000 --maxi-batch 100 \
    --early-stopping 10 \
    --valid-freq 5000 --save-freq 5000 --disp-freq 500 \
    --valid-metrics cross-entropy perplexity bleu \
    --valid-sets ${data_path}/dev.${src} ${data_path}/dev.${tgt} \
    --valid-translation-output ${model_folder}/valid.${src}-${tgt}.output --quiet-translation \
    --valid-mini-batch 64 \
    --beam-size 6 --normalize 0.6 \
    --log ${model_folder}/train.log --valid-log ${model_folder}/valid.log \
    --enc-depth 2 --dec-depth 2 \
    --transformer-heads 8 \
    --transformer-postprocess-emb d \
    --transformer-postprocess dan \
    --transformer-dropout 0.3 --label-smoothing 0.1 \
    --learn-rate 0.0003 --lr-warmup 0 --lr-decay-inv-sqrt 16000 --lr-report \
    --clip-norm 5 \
    --tied-embeddings \
    --devices 0 --sync-sgd --seed 1111 \
    --exponential-smoothing \
    --dump-config > ${model_folder}/${src}-${tgt}.config.yml

time marian -c ${model_folder}/${src}-${tgt}.config.yml 2>&1 | tee ${model_folder}/transformer-${src}-${tgt}.log

Writing ./transformer.phmy.sh


In [ ]:
!chmod +x ./transformer.phmy.sh

ဒီတခါတော့ architecture ကို transformer အဖြစ် ပြောင်းလိုက်ပါတယ်။  

```bash
--type transformer
```

## Training Transformer Model for Phoneme to Grapheme Translation

In [ ]:
%pwd

'/content/marian-demo'

In [ ]:
!ldd /content/marian-bin-gpu/marian

	linux-vdso.so.1 (0x00007fff1ddc1000)
	libcublasLt.so.12 => /usr/local/cuda/lib64/libcublasLt.so.12 (0x00007ab036800000)
	libcurand.so.10 => /usr/local/cuda/lib64/libcurand.so.10 (0x00007ab02dc00000)
	libcusparse.so.12 => /usr/local/cuda/lib64/libcusparse.so.12 (0x00007ab016800000)
	libcublas.so.12 => /usr/local/cuda/lib64/libcublas.so.12 (0x00007ab00f600000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x00007ab00f3d4000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x00007ab068b14000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x00007ab068af4000)
	libc.so.6 => /lib/x86_64-linux-gnu/libc.so.6 (0x00007ab00f1ab000)
	/lib64/ld-linux-x86-64.so.2 (0x00007ab07525e000)
	librt.so.1 => /lib/x86_64-linux-gnu/librt.so.1 (0x00007ab0367fb000)
	libpthread.so.0 => /lib/x86_64-linux-gnu/libpthread.so.0 (0x00007ab0367f6000)
	libdl.so.2 => /lib/x86_64-linux-gnu/libdl.so.2 (0x00007ab0367f1000)
	libnvJitLink.so.12 => /usr/local/cuda/lib64/libnvJitLink.so.12 (0x00007ab0096c3000)


In [ ]:
!./transformer.phmy.sh

[2026-08-07 03:08:05] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-08-07 03:08:05] [marian] Running on 389c9429af10 as process 1639 with command line:
[2026-08-07 03:08:05] [marian] marian -c model.transformer.phmy/ph-my.config.yml
[2026-08-07 03:08:05] [config] after: 0e
[2026-08-07 03:08:05] [config] after-batches: 0
[2026-08-07 03:08:05] [config] after-epochs: 0
[2026-08-07 03:08:05] [config] all-caps-every: 0
[2026-08-07 03:08:05] [config] allow-unk: false
[2026-08-07 03:08:05] [config] authors: false
[2026-08-07 03:08:05] [config] beam-size: 6
[2026-08-07 03:08:05] [config] bert-class-symbol: "[CLS]"
[2026-08-07 03:08:05] [config] bert-mask-symbol: "[MASK]"
[2026-08-07 03:08:05] [config] bert-masking-fraction: 0.15
[2026-08-07 03:08:05] [config] bert-sep-symbol: "[SEP]"
[2026-08-07 03:08:05] [config] bert-train-type-embeddings: true
[2026-08-07 03:08:05] [config] bert-type-vocab-size: 2
[2026-08-07 03:08:05] [config] build-info: ""
[2026-08-07 03:08:05] [config

### Save Trained Model to Google Drive

In [ ]:
import shutil, os
drive_save_path = '/content/drive/MyDrive/marian-results/transformer.phmy'
os.makedirs(os.path.dirname(drive_save_path), exist_ok=True)
shutil.copytree('./model.transformer.phmy', drive_save_path, dirs_exist_ok=True)
print(f'✅ Saved to {drive_save_path}')


✅ Saved to /content/drive/MyDrive/marian-results/transformer.phmy


## Checking Models  

In [ ]:
!ls ./model.transformer.phmy

model.iter10000.npz  model.iter45000.npz      model.npz.progress.yml
model.iter15000.npz  model.iter50000.npz      model.npz.yml
model.iter20000.npz  model.iter5000.npz       ph-my.config.yml
model.iter25000.npz  model.iter55000.npz      train.log
model.iter30000.npz  model.npz		      transformer-ph-my.log
model.iter35000.npz  model.npz.decoder.yml    valid.log
model.iter40000.npz  model.npz.optimizer.npz  valid.ph-my.output


In [ ]:
!cat ./model.transformer.phmy/valid.log

[2026-08-07 03:11:45] [valid] Ep. 80 : Up. 5000 : cross-entropy : 1.82078 : new best
[2026-08-07 03:11:45] [valid] Ep. 80 : Up. 5000 : perplexity : 1.60597 : new best
[2026-08-07 03:11:45] [valid] First sentence's tokens as scored:
[2026-08-07 03:11:45] [valid] DefaultVocab keeps original segments for scoring
[2026-08-07 03:11:45] [valid]   Hyp: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-08-07 03:11:45] [valid]   Ref: ဥတ် တ ရ ဖ လ ဂု နီ
[2026-08-07 03:11:46] [valid] Ep. 80 : Up. 5000 : bleu : 76.3169 : new best
[2026-08-07 03:15:26] [valid] Ep. 159 : Up. 10000 : cross-entropy : 2.02304 : stalled 1 times (last best: 1.82078)
[2026-08-07 03:15:26] [valid] Ep. 159 : Up. 10000 : perplexity : 1.69275 : stalled 1 times (last best: 1.60597)
[2026-08-07 03:15:27] [valid] Ep. 159 : Up. 10000 : bleu : 75.753 : stalled 1 times (last best: 76.3169)
[2026-08-07 03:19:06] [valid] Ep. 239 : Up. 15000 : cross-entropy : 2.18463 : stalled 2 times (last best: 1.82078)
[2026-08-07 03:19:06] [valid] Ep. 239 : Up. 15000 : perp

## Testing Transformer Model for Phoneme to Grapheme Translation

Marian ရဲ့ decoder command ကိုလည်း လေ့လာကြည့်ပါ။  

In [ ]:
!marian-decoder --help

Marian: Fast Neural Machine Translation in C++
Usage: marian-decoder [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=512                Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-time-zon

In [ ]:
!time marian-decoder -m ./model.transformer.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 < ./g2p-par/test.ph > ./transformer.phmy.hyp.txt

[2026-08-07 03:48:39] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-08-07 03:48:39] [marian] Running on 389c9429af10 as process 12215 with command line:
[2026-08-07 03:48:39] [marian] marian-decoder -m ./model.transformer.phmy/model.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0
[2026-08-07 03:48:39] [config] alignment: ""
[2026-08-07 03:48:39] [config] allow-special: false
[2026-08-07 03:48:39] [config] allow-unk: false
[2026-08-07 03:48:39] [config] authors: false
[2026-08-07 03:48:39] [config] beam-size: 12
[2026-08-07 03:48:39] [config] bert-class-symbol: "[CLS]"
[2026-08-07 03:48:39] [config] bert-mask-symbol: "[MASK]"
[2026-08-07 03:48:39] [config] bert-masking-fraction: 0.15
[2026-08-07 03:48:39] [config] bert-sep-symbol: "[SEP]"
[2026-08-07 03:48:39] [config] bert-train-type-embeddings: true
[2026-08-07 03:48:39] [config] bert-type-vocab-size: 2
[2026-08-07 03:48:39] [config] best-deep: false
[2026-08-07 03:48:39] [config] build-

**Marian က training/testing အတွက် အရမ်းမြန်ပါတယ်။ အဲဒါကြောင့် ဆရာ ကြိုက်တယ်။**

## Evaluation

### Colab Adaptation: get `multi-bleu.perl`

In [ ]:
!wget -nc -q https://raw.githubusercontent.com/moses-smt/mosesdecoder/master/scripts/generic/multi-bleu.perl -O ./multi-bleu.perl


In [ ]:
!perl ./multi-bleu.perl ./g2p-par/test.my < ./transformer.phmy.hyp.txt


BLEU = 76.32, 87.4/78.2/72.5/68.7 (BP=0.999, ratio=0.999, hyp_len=8040, ref_len=8047)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.


Sequence to Sequence နဲ့ Transformer မော်ဒယ်နှစ်ရဲ့ ရလဒ်ကို နှိုင်းယှဉ်ကြည့်တဲ့အခါမှာ Sequence to Sequence မော်ဒယ်ရဲ့ ရလဒ်က ပိုကောင်းတာကို တွေ့ရပါလိမ့်မယ်။  

## Task 4: Improving the Transformer Baseline

I don't have a local GPU, so all training and experiments ran on Google Colab's free-tier GPU, which comes with limited and unpredictable runtime. That ruled out any approach needing multiple full retraining runs — I needed something reliable in one shot.

For Seq2Seq, I first tried fine-tuning the already-converged model with label smoothing and LR warmup — it made BLEU worse, since the extra loss term conflicted with weights that were already confident. Retraining a finished model isn't reliably safe, and re-running it to fix would burn more of my limited GPU time.

So for Transformer, I skipped fine-tuning and went straight to checkpoint-based improvement — no additional training needed.

Marian saves a checkpoint every `--save-freq` updates (see `model.transformer.phmy/` for the `model.iterXXXX.npz` files), not just the final `model.npz`. From `valid.log`, the best validation BLEU (76.7046) was at `model.iter15000.npz`, not the final checkpoint.

Rather than use just that one checkpoint, I decoded with two checkpoints together — `model.iter15000.npz` and `model.iter35000.npz` — using Marian's ensemble decoding (pass both with `-m`), plus `--beam-size 6 --normalize 0.6` to match the teacher's original validation settings (the baseline decode above didn't use these).

In [ ]:
!grep -i " bleu : " ./model.transformer.phmy/valid.log | sed -E 's/.*Up\. ([0-9]+).*bleu : ([0-9.]+).*/\2 \1/' | sort -n -r | head -5

76.7046 15000
76.3941 35000
76.3766 55000
76.3169 5000
75.992 50000


In [ ]:
!time marian-decoder -m ./model.transformer.phmy/model.iter15000.npz ./model.transformer.phmy/model.iter35000.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 --beam-size 6 --normalize 0.6 < ./g2p-par/test.ph > ./transformer.phmy.ensemble.hyp.txt
!perl ./multi-bleu.perl ./g2p-par/test.my < ./transformer.phmy.ensemble.hyp.txt

[2026-08-07 04:12:30] [marian] Marian v1.12.0 65bf82ff 2023-02-21 09:56:29 -0800
[2026-08-07 04:12:30] [marian] Running on 389c9429af10 as process 18303 with command line:
[2026-08-07 04:12:30] [marian] marian-decoder -m ./model.transformer.phmy/model.iter15000.npz ./model.transformer.phmy/model.iter35000.npz -v ./g2p-par/vocab/vocab.ph.yml ./g2p-par/vocab/vocab.my.yml --devices 0 --beam-size 6 --normalize 0.6
[2026-08-07 04:12:30] [config] alignment: ""
[2026-08-07 04:12:30] [config] allow-special: false
[2026-08-07 04:12:30] [config] allow-unk: false
[2026-08-07 04:12:30] [config] authors: false
[2026-08-07 04:12:30] [config] beam-size: 6
[2026-08-07 04:12:30] [config] bert-class-symbol: "[CLS]"
[2026-08-07 04:12:30] [config] bert-mask-symbol: "[MASK]"
[2026-08-07 04:12:30] [config] bert-masking-fraction: 0.15
[2026-08-07 04:12:30] [config] bert-sep-symbol: "[SEP]"
[2026-08-07 04:12:30] [config] bert-train-type-embeddings: true
[2026-08-07 04:12:30] [config] bert-type-vocab-size: 2
[

### Evaluate and compare to the baseline BLEU score above


| Model | BLEU |
|---|---|
| Baseline (final `model.npz`) | 76.32 |
| Improved (ensemble of `model.iter15000.npz` + `model.iter35000.npz`, beam-size 6, normalize 0.6) | 77.25 |

Using an ensemble of two checkpoints instead of the final model — one from the peak validation BLEU (iter15000) and one from later in training (iter35000) — along with the teacher's original beam/normalize decode settings, improves the test BLEU score with zero additional training. This is a step further than the Seq2Seq approach, which just swapped in a single better checkpoint; here, ensembling two checkpoints was what actually beat the baseline, since the single-checkpoint validation BLEU gain didn't reliably carry over to the test set.
